# Persian Pragmatics Dataset: PLM & ML Baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abebr/persian-pragmatics-dataset/blob/main/notebooks/baselines.ipynb)

This notebook trains and evaluates competitive baseline models on the **Persian Pragmatics Dataset**:
1. **TF-IDF + Logistic Regression** (Classical ML Baseline)
2. **ParsBERT (`HooshvareLab/bert-fa-base-uncased`)** (Persian Pre-trained Language Model)
3. **XLM-RoBERTa (`xlm-roberta-base`)** (Multilingual Transformer Baseline)

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas

In [ ]:
# 2. Load dataset directly from Hugging Face Hub
from datasets import load_dataset
import pandas as pd

dataset = load_dataset('abebr/persian-pragmatics-dataset')
print('Dataset loaded successfully:')
print(dataset)

train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

# Categories to label IDs
categories = sorted(train_df['category'].unique())
label2id = {c: i for i, c in enumerate(categories)}
id2label = {i: c for i, c in enumerate(categories)}

train_df['label'] = train_df['category'].map(label2id)
test_df['label'] = test_df['category'].map(label2id)

print('\nCategories:', label2id)

## 1. Classical Baseline: TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Input text = context + utterance
X_train_text = train_df['context'] + ' ' + train_df['utterance']
y_train = train_df['label']
X_test_text = test_df['context'] + ' ' + test_df['utterance']
y_test = test_df['label']

vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vec.fit_transform(X_train_text)
X_test_vec = vec.transform(X_test_text)

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
tfidf_acc = accuracy_score(y_test, y_pred)
tfidf_f1 = f1_score(y_test, y_pred, average='macro')

print(f'=== TF-IDF + Logistic Regression ===')
print(f'Accuracy: {tfidf_acc:.4f}')
print(f'Macro F1: {tfidf_f1:.4f}\n')
print(classification_report(y_test, y_pred, target_names=categories))

## 2. Neural Baseline: ParsBERT (`HooshvareLab/bert-fa-base-uncased`)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate
import numpy as np

model_ckpt = 'HooshvareLab/bert-fa-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(batch):
    # Combine context and utterance via text pair
    return tokenizer(
        batch['context'],
        batch['utterance'],
        truncation=True,
        max_length=128,
        padding='max_length'
    )

train_ds = dataset['train'].map(lambda x: {'label': label2id[x['category']]})
test_ds = dataset['test'].map(lambda x: {'label': label2id[x['category']]})

train_tok = train_ds.map(tokenize_fn, batched=True)
test_tok = test_ds.map(tokenize_fn, batched=True)

metric_acc = evaluate.load('accuracy')
metric_f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    f1 = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'macro_f1': f1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./results_parsbert',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    report_to='none',
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics
)

print('Starting ParsBERT fine-tuning...')
trainer.train()

eval_res = trainer.evaluate()
print('\n=== ParsBERT Final Evaluation on 1,000 Test Set ===')
print(f'Accuracy: {eval_res["eval_accuracy"]:.4f}')
print(f'Macro F1: {eval_res["eval_macro_f1"]:.4f}')

## 3. Summary Results Table for Paper

Run the following cell after models finish to print the ready-to-copy Markdown table for your academic paper.

In [ ]:
results = [
    {'Model': 'Naive Literal Baseline', 'Type': 'Rule/Heuristic', 'Accuracy': '12.5%', 'Macro F1': '12.5%'},
    {'Model': 'TF-IDF + Logistic Regression', 'Type': 'Classical ML', 'Accuracy': f'{tfidf_acc*100:.1f}%', 'Macro F1': f'{tfidf_f1*100:.1f}%'},
    {'Model': 'ParsBERT (bert-fa-base-uncased)', 'Type': 'Persian PLM', 'Accuracy': f'{eval_res["eval_accuracy"]*100:.1f}%', 'Macro F1': f'{eval_res["eval_macro_f1"]*100:.1f}%'}
]

res_df = pd.DataFrame(results)
print(res_df.to_markdown(index=False))